___
# <center>Regressão linear múltipla e regressão logística</center>
___

## Aula 20

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * ler a saída de uma regressão no `statsmodels`: coeficiente, intervalo,
   valor-p e $R^2$;
 * interpretar cada coeficiente de uma regressão múltipla **mantidas as demais
   variáveis**, e explicar por que ele muda quando outra variável entra;
 * interpretar variáveis de sim ou não e de categorias (dummies);
 * ajustar uma regressão logística e ler $e^{b}$ como **razão de chances**;
 * calcular a probabilidade prevista para um perfil de processo.

**Primeira hora:** regressão linear múltipla, retomando a aula 19 com calma.
**Segunda hora:** regressão logística, na mesma base.

___
<div id="indice"></div>

## Índice

- [A base](#base)
- **Primeira hora: regressão linear**
  - [1. Regressão simples](#simples)
  - [2. Valor-p e R² respondem perguntas diferentes](#r2)
  - [3. Regressão múltipla](#multipla)
  - [4. Variáveis de sim ou não](#binaria)
  - [5. Variáveis com categorias](#categorias)
  - [6. Exercício: as custas do processo](#ex-linear)
- **Segunda hora: regressão logística**
  - [7. Desfecho de sim ou não](#sim-nao)
  - [8. Chance e razão de chances](#chance)
  - [9. Regressão logística](#logistica)
  - [10. Probabilidade prevista](#previsao)
  - [11. Exercício: a perícia contábil ajuda o autor?](#ex-logistica)

___
<div id="base"></div>

# A base

**Atenção: a base é simulada.** Ela imita a carteira de ações revisionais de
financiamento de veículo que vimos com o BTG, mas cada linha foi sorteada no
computador, com efeitos escolhidos para a aula. Por isso ela é pública e vem
direto da internet, sem baixar nada. Os resultados ilustram os métodos e **não
dizem nada sobre processos reais**.

Cada linha é um processo já julgado.

| coluna | o que é |
|---|---|
| `tribunal` | TJSP, TJMG, TJRS, TJPR ou TJBA |
| `juizado` | 1 se tramitou no juizado especial, 0 se na vara comum |
| `escritorio_massa` | 1 se o advogado do autor é de escritório com muitas ações iguais |
| `idade_autor` | idade do autor, em anos |
| `valor_financiado` | valor financiado no contrato, em R$ mil |
| `taxa_juros` | taxa de juros do contrato, em % ao mês |
| `valor_causa` | valor da causa (o que o autor pediu), em R$ mil |
| `pericia` | 1 se houve perícia contábil |
| `sentenca_favoravel` | 1 se a sentença foi favorável ao autor, 0 se desfavorável |
| `valor_condenacao` | quanto o banco foi condenado a pagar, em R$ mil; vazio (`NaN`) quando a sentença foi desfavorável |
| `custas` | custas e despesas do processo, em R$ mil |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

processos = pd.read_csv("https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/processos_simulados.csv")
processos.head()

In [ ]:
processos.describe().round(1)

`describe()` resume cada coluna numérica. Repare em duas coisas:

* nas colunas de 0 e 1, a média é a proporção de 1: a média de
  `sentenca_favoravel` é a fração de sentenças favoráveis ao autor;
* o `count` de `valor_condenacao` é menor que 1.000, porque ela fica vazia nas
  sentenças desfavoráveis. Nas regressões com essa coluna, o `statsmodels`
  deixa essas linhas de fora sozinho: o `No. Observations` da saída mostra
  quantas ficaram.

[Volta ao Índice](#indice)

___
<div id="simples"></div>

# Primeira hora: regressão linear

## 1. Regressão simples

A pergunta da primeira hora: **nas sentenças favoráveis, o que está associado
ao valor da condenação?** Começamos com uma variável só: o valor da causa.

In [ ]:
processos.plot.scatter(x="valor_causa", y="valor_condenacao", alpha=0.3)

In [ ]:
modelo1 = smf.ols("valor_condenacao ~ ____", data=processos).fit()
modelo1.summary()

Onde olhar na saída:

* **`No. Observations`:** 568, os processos com sentença favorável.
* **`coef` de `valor_causa`:** quanto a condenação muda, em média, quando o
  valor da causa aumenta 1 unidade. As duas colunas estão em R$ mil, então 0,310
  quer dizer: **cada R$ 1.000 a mais na causa estão associados a cerca de R$ 310
  a mais de condenação**.
* **`[0.025  0.975]`:** o intervalo de 95% do coeficiente, de R$ 288 a R$ 332.
* **`P>|t|`:** o valor-p. Perto de zero: se não houvesse associação nenhuma,
  seria muito difícil ver um coeficiente desse tamanho por acaso.
* **`Intercept`:** a condenação prevista para uma causa de R$ 0. Fora dos
  dados, sem leitura prática.
* **`R-squared`, no alto à direita:** 0,574. A reta explica 57% da variação da
  condenação entre os processos.

In [ ]:
# a previsão da reta para uma causa de R$ 40 mil
modelo1.predict(pd.DataFrame({"valor_causa": [40]}))

[Volta ao Índice](#indice)

___
<div id="r2"></div>

## 2. Valor-p e R² respondem perguntas diferentes

Mais dois modelos simples, cada um com uma explicativa. Compare o coeficiente,
o valor-p e o $R^2$.

In [ ]:
modelo_idade = smf.ols("valor_condenacao ~ idade_autor", data=processos).fit()
modelo_idade.summary()

In [ ]:
modelo_juros = smf.ols("valor_condenacao ~ taxa_juros", data=processos).fit()
modelo_juros.summary()

| explicativa | o coeficiente é diferente de zero? | $R^2$ |
|---|---|---|
| valor da causa | sim (valor-p < 0,001) | 0,57: explica muito |
| idade do autor | não (valor-p = 0,68) | 0,000: não explica nada |
| taxa de juros | sim (valor-p < 0,001) | 0,05: explica pouco |

* O **valor-p** pergunta: *existe* associação, ou o que vemos pode ser acaso?
* O **$R^2$** pergunta: *quanto* da variação da condenação a variável explica?

A taxa de juros está associada à condenação, mas explica pouco: entre
processos com a mesma taxa, as condenações ainda variam muito. **R² baixo não
quer dizer coeficiente inútil.**

[Volta ao Índice](#indice)

___
<div id="multipla"></div>

## 3. Regressão múltipla

Antes de pôr várias variáveis no modelo, a correlação entre elas.

In [ ]:
colunas = ["valor_condenacao", "valor_causa", "valor_financiado", "taxa_juros", "idade_autor"]
processos[colunas].corr().round(2)

O valor da causa tem correlação de 0,76 com o valor financiado: contratos
maiores geram pedidos maiores. E o financiado também está associado à
condenação. Então parte do coeficiente da causa, no modelo simples, é "tamanho
do contrato".

In [ ]:
modelo2 = smf.ols("valor_condenacao ~ valor_causa + ____", data=processos).fit()
modelo2.summary()

O coeficiente da causa caiu de 0,310 para 0,258. Agora ele vale **mantido o
valor financiado**: entre processos com o mesmo valor financiado, cada R$ 1.000
a mais na causa estão associados a cerca de R$ 258 a mais de condenação.

Agora com todas as variáveis numéricas:

In [ ]:
modelo3 = smf.ols("valor_condenacao ~ valor_causa + valor_financiado + taxa_juros + idade_autor",
                  data=processos).fit()
modelo3.summary()

**Como relatar cada coeficiente:** a unidade, o "mantidas as demais" e o
intervalo.

* `valor_causa`: "Mantidos o valor financiado, a taxa de juros e a idade do
  autor, cada R$ 1.000 a mais na causa estão associados a cerca de R$ 234 a mais
  de condenação (IC 95%: R$ 198 a R$ 269)."
* `taxa_juros`: cada ponto percentual a mais de juros ao mês, mantidas as
  demais, está associado a cerca de R$ 1.800 a mais de condenação (IC 95%: R$
  810 a R$ 2.790).
* `idade_autor`: valor-p de 0,57 e intervalo que inclui zero. **Não há
  evidência** de associação com a condenação.

**R² e R² ajustado.** O $R^2$ foi de 0,585 para 0,595. Ele nunca cai quando uma
variável entra, mesmo que ela não ajude; o ajustado desconta o número de
variáveis.

[Volta ao Índice](#indice)

___
<div id="binaria"></div>

## 4. Variáveis de sim ou não

`juizado` vale 0 ou 1. Na regressão, o coeficiente de uma variável assim é a
**diferença de médias** entre os dois grupos: é o teste t, escrito como
regressão.

In [ ]:
processos.groupby("juizado")["valor_condenacao"].mean().round(2)

In [ ]:
sem_causa = smf.ols("valor_condenacao ~ juizado", data=processos).fit()
sem_causa.summary()

O intercepto (12,31) é a média da vara comum, e o coeficiente (-4,60) é a
diferença: no juizado, a condenação é, em média, cerca de R$ 4.600 menor.

Mas as causas do juizado são menores (ele só aceita causas até 40 salários
mínimos). E se compararmos processos com o mesmo valor da causa?

In [ ]:
com_causa = smf.ols("valor_condenacao ~ juizado + ____", data=processos).fit()
com_causa.summary()

Com o valor da causa no modelo, a diferença do juizado vira R$ 480 **a mais**,
com valor-p de 0,27 e intervalo que inclui zero: **some**. Era o tamanho da
causa que aparecia na comparação simples.

In [ ]:
processos.groupby("juizado")[["valor_causa", "valor_condenacao"]].mean().round(1)

[Volta ao Índice](#indice)

___
<div id="categorias"></div>

## 5. Variáveis com categorias

O tribunal tem cinco categorias. Na regressão, ele vira **quatro colunas de 0 e
1** (dummies), e o tribunal que fica de fora é a **referência**. Escolhemos o
TJSP, o maior.

In [ ]:
processos.groupby("tribunal")["valor_condenacao"].agg(["count", "mean"]).round(2)

In [ ]:
modelo_trib = smf.ols('valor_condenacao ~ C(tribunal, Treatment("____"))', data=processos).fit()
modelo_trib.summary()

* O **intercepto** (10,75) é a média do TJSP.
* Cada coeficiente é a **diferença para o TJSP**: no TJRS a condenação é, em
  média, R$ 4.200 maior (valor-p < 0,001); no TJBA, R$ 3.220 menor (valor-p <
  0,001).
* TJMG e TJPR têm valor-p acima de 0,6: não se distinguem do TJSP.

Comparar a média de várias categorias de uma vez é a ANOVA da aula 17, aqui
escrita como regressão.

[Volta ao Índice](#indice)

___
<div id="ex-linear"></div>

## 6. Exercício: as custas do processo

Agora a variável resposta é `custas`: custas e despesas do processo, em R$ mil.
Ela existe nos 1.000 processos. Em duplas.

In [ ]:
custas1 = smf.ols("custas ~ ____", data=processos).fit()
custas1.summary()

In [ ]:
custas2 = smf.ols('custas ~ juizado + pericia + C(tribunal, Treatment("TJSP"))',
                  data=processos).fit()
custas2.summary()

**a)** No primeiro modelo, o valor da causa está associado às custas? Quanto
ele explica?

**b)** No segundo, escreva a frase do coeficiente da perícia e a do juizado,
com unidade e "mantidas as demais".

**c)** Qual tribunal tem as custas mais altas, mantidos juizado e perícia?
Algum não se distingue do TJSP?

**d)** Compare os dois $R^2$. Qual modelo explica melhor as custas?

*(escreva aqui)*

[Volta ao Índice](#indice)

___
<div id="sim-nao"></div>

# Segunda hora: regressão logística

## 7. Desfecho de sim ou não

A pergunta da segunda hora: **o que está associado a uma sentença favorável ao
autor?** `sentenca_favoravel` é 1 ou 0. A regressão linear roda, mas não sabe
que a resposta é uma probabilidade.

In [ ]:
reta = smf.ols("sentenca_favoravel ~ taxa_juros", data=processos).fit()
reta.predict(pd.DataFrame({"taxa_juros": [1, 2, 3, 4]})).round(3)

Para juros de 3% ao mês, que existem na base, a reta já prevê uma
probabilidade de sentença favorável **acima de 100%** (102%); para 4%, 148%. A
regressão logística modela a chance numa escala em que a curva fica sempre
entre 0 e 1.

[Volta ao Índice](#indice)

___
<div id="chance"></div>

## 8. Chance e razão de chances

$$\text{chance} = \frac{p}{1 - p}$$

Probabilidade de 80% é chance de 4: 4 sentenças favoráveis para cada
desfavorável.

In [ ]:
pd.crosstab(processos["escritorio_massa"], processos["sentenca_favoravel"], margins=True)

In [ ]:
chance_massa = 162 / 189             # favoráveis / desfavoráveis, com escritório de massa
chance_demais = ____ / ____          # favoráveis / desfavoráveis, nos demais
print(f"chance com escritório de massa: {chance_massa:.2f}")
print(f"chance nos demais: {chance_demais:.2f}")
print(f"razão de chances: {chance_massa / chance_demais:.2f}")

Com escritório de massa, a chance de sentença favorável é **0,51 vez** a dos
demais: mais ou menos a metade. Em probabilidade: 46% contra 63%.

[Volta ao Índice](#indice)

___
<div id="logistica"></div>

## 9. Regressão logística

$$\log\left(\frac{p}{1-p}\right) = a + b_1 x_1 + b_2 x_2 + \dots$$

O coeficiente $b$ está na escala do log da chance. **$e^{b}$ é a razão de
chances**: quantas vezes a chance é multiplicada quando $x$ aumenta 1 unidade,
mantidas as demais.

Com o escritório de massa sozinho, o modelo reproduz a conta da tabela:

In [ ]:
logit1 = smf.logit("sentenca_favoravel ~ escritorio_massa", data=processos).fit()
np.exp(logit1.____)

Agora com várias variáveis:

In [ ]:
logit2 = smf.logit("sentenca_favoravel ~ taxa_juros + escritorio_massa + juizado + idade_autor",
                   data=processos).fit()
logit2.summary()

In [ ]:
# razões de chances e intervalos de 95%
print(np.exp(logit2.params).round(3))
print(np.exp(logit2.conf_int()).round(3))

**Como relatar:**

* `taxa_juros`: mantidos o escritório, o juizado e a idade, cada ponto
  percentual a mais de juros ao mês **multiplica** a chance de sentença
  favorável por **7,19** (IC 95%: 4,70 a 11,01). Juro alto é o que o juiz
  reconhece como abusivo.
* `escritorio_massa`: mantidos os juros, o juizado e a idade, a chance com
  escritório de massa é **0,71 vez** a dos demais (IC 95%: 0,54 a 0,94), cerca
  de 30% menor. Sozinho era 0,51: os clientes de escritório de massa têm
  contratos com juros menores (1,88% contra 2,08% ao mês), e parte da diferença
  vinha disso.
* `juizado`: razão de 0,96, valor-p de 0,80. Sem evidência de diferença para a
  vara comum.
* `idade_autor`: razão de 1,003, valor-p de 0,62. Sem evidência de associação.

Razão de chances **acima de 1** aumenta a chance; **abaixo de 1** diminui;
**igual a 1** não muda nada.

In [ ]:
processos.groupby("escritorio_massa")["taxa_juros"].mean().round(2)

[Volta ao Índice](#indice)

___
<div id="previsao"></div>

## 10. Probabilidade prevista

O modelo devolve a probabilidade de sentença favorável para qualquer perfil de
processo.

In [ ]:
perfis = pd.DataFrame({
    "taxa_juros":       [2.5, 2,  2,  1.5],
    "escritorio_massa": [0,   0,  1,  1],
    "juizado":          [0,   0,  0,  0],
    "idade_autor":      [40,  40, 40, 40],
})
perfis["prob_favoravel"] = logit2.____(perfis).round(3)
perfis

Juros de 2,5% ao mês, sem escritório de massa: 80%. Com 2%: 60%. Os mesmos 2%
com escritório de massa: 52%. E 1,5% com escritório de massa: 29%.

A probabilidade prevista é o que vira decisão: na aula 21 vamos escolher um
**ponto de corte** e contar os erros de cada lado.

[Volta ao Índice](#indice)

___
<div id="ex-logistica"></div>

## 11. Exercício: a perícia contábil ajuda o autor?

Em duplas. Siga as etapas e escreva o relatório no fim.

**1. Exploratória.** A proporção de sentenças favoráveis com e sem perícia, com
o número de processos.

In [ ]:
processos.groupby("____")["sentenca_favoravel"].agg(["size", "mean"]).round(3)

**2. Modelo.** Regressão logística com a perícia, ajustada pelos juros, pelo
escritório de massa, pelo juizado e pelo tribunal (referência: TJSP). Calcule
as razões de chances.

In [ ]:
logit3 = smf.logit('sentenca_favoravel ~ pericia + taxa_juros + escritorio_massa + juizado'
                   ' + C(tribunal, Treatment("TJSP"))', data=processos).fit()
np.exp(____).round(3)

In [ ]:
np.exp(logit3.conf_int()).round(3)

**3. Previsão.** A probabilidade de sentença favorável num processo da vara
comum do TJSP, com juros de 2% ao mês e sem escritório de massa, com e sem
perícia.

In [ ]:
novos = pd.DataFrame({"pericia": [1, 0], "taxa_juros": [2, 2], "escritorio_massa": [0, 0],
                      "juizado": [0, 0], "tribunal": ["TJSP", "TJSP"]})
logit3.predict(novos).round(3)

**4. Relatório**, em três frases, na célula abaixo: a pergunta; o resultado,
com a razão de chances, o intervalo e "mantidos..."; e **uma** limitação (o que
o modelo não permite concluir).

*(escreva aqui)*

[Volta ao Índice](#indice)